# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshiniChebrolu/Flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

I will use Multinomial Logistic Regression as the first capstone model.

The Growth lane is concerned with distinguishing between three possible action directions: REFRESH, CTR_FIX, and MONITOR. Logistic Regression is appropriate as an interpretable multiclass baseline because it produces a separate probability for each class and allows the contribution of input features to be inspected.

I will prefer this simple model before considering more complex models. This keeps the comparison focused on whether a learned model provides useful signal beyond the Week-4 rule, rather than rewarding complexity alone.

The model will use only information available at the observation point and will exclude future outcome fields from the feature set.


In [18]:
# ML-08 — Section 1: Load and inspect dataset

import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))
df = pd.read_csv(filename)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Saving content_refresh_anonymized (2).csv to content_refresh_anonymized (2) (1).csv
Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [19]:
# Check the fields relevant to the modeling task

print("Target candidate: trend_direction")

if "trend_direction" in df.columns:
    print("\ntrend_direction values:")
    print(df["trend_direction"].value_counts(dropna=False))
else:
    print("\ntrend_direction is not present in the dataset.")

print("\nW04 baseline-related columns:")

for col in [
    "days_since_last_update",
    "avg_position",
    "ctr",
    "action",
    "action_score"
]:
    if col in df.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col}")

Target candidate: trend_direction

trend_direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

W04 baseline-related columns:
✓ days_since_last_update
✓ avg_position
✓ ctr
✗ action
✗ action_score


## 2. Split design

I will use a stratified 80/20 train-test split because the modeling task contains multiple outcome classes. Stratification helps preserve the observed class proportions in both the training and test sets.

The test set will remain separate from model training and will be used for evaluating the Week-5 model. The same held-out observations will be used when comparing the model with the Week-4 baseline.

Future outcome fields will not be included as model inputs. A fixed random state will be used so that the split is reproducible.


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Section 2: Split design

from sklearn.model_selection import train_test_split

# Target for the modeling task
target_col = "trend_direction"

# Check that the target exists
if target_col not in df.columns:
    raise KeyError(
        f"{target_col} is not present in the dataset. "
        f"Available columns: {df.columns.tolist()}"
    )

# Remove rows where the target is missing
model_df = df[df[target_col].notna()].copy()

print("Rows available for modeling:", len(model_df))
print("\nTarget distribution:")
print(model_df[target_col].value_counts(dropna=False))

Rows available for modeling: 30000

Target distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [21]:
# Fields that must not be used as model inputs

excluded_cols = [
    "trend_direction",
    "trend_pct",
    "action",
    "action_score"
]

# Remove identifier fields if they exist
identifier_cols = [
    "url",
    "page_url",
    "client_name"
]

excluded_cols += [
    col for col in identifier_cols
    if col in model_df.columns
]

feature_cols = [
    col for col in model_df.columns
    if col not in excluded_cols
]

X = model_df[feature_cols].copy()
y = model_df[target_col].copy()

print("Number of features:", len(feature_cols))
print("\nExcluded columns:")
print(excluded_cols)

print("\nFeatures:")
print(feature_cols)


Number of features: 42

Excluded columns:
['trend_direction', 'trend_pct', 'action', 'action_score']

Features:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


In [22]:
# Create reproducible stratified train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training rows: 24000
Test rows: 6000

Training class distribution:
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64

Test class distribution:
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.074
flat      0.038
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

I will train the Logistic Regression model using the training portion of the dataset and evaluate it on the held-out test set.

The Week-4 baseline will be kept separate from the model inputs and will be used only for comparison. Both approaches will be evaluated on the same test observations.

Macro F1 will be used as the primary metric because the task contains multiple outcome classes and gives equal importance to each class. Accuracy will also be reported as a secondary metric.

The comparison is intended to measure whether the learned model provides useful predictive signal beyond the transparent Week-4 baseline.


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Section 3: Prepare preprocessing

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Identify numeric and categorical features
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 29
Categorical features: 13


In [24]:
# Numeric preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Logistic Regression model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Train
model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [25]:
# Generate predictions
model_pred = model.predict(X_test)

print("First 10 predictions:")
print(model_pred[:10])


First 10 predictions:
['up' 'down' 'down' 'new' 'down' 'new' 'down' 'down' 'stable' 'stable']


In [26]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

model_accuracy = accuracy_score(
    y_test,
    model_pred
)

model_macro_f1 = f1_score(
    y_test,
    model_pred,
    average="macro"
)

print("Model Accuracy:", round(model_accuracy, 4))
print("Model Macro F1:", round(model_macro_f1, 4))

print("\nClassification Report:")
print(classification_report(y_test, model_pred))

Model Accuracy: 0.757
Model Macro F1: 0.6644

Classification Report:
              precision    recall  f1-score   support

        down       0.76      0.95      0.84      3252
        flat       0.59      0.49      0.53       231
         new       0.72      0.80      0.76       447
      stable       0.75      0.49      0.60      1192
          up       0.82      0.46      0.59       878

    accuracy                           0.76      6000
   macro avg       0.73      0.64      0.66      6000
weighted avg       0.76      0.76      0.74      6000



In [27]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

model_accuracy = accuracy_score(
    y_test,
    model_pred
)

model_macro_f1 = f1_score(
    y_test,
    model_pred,
    average="macro"
)

print("Model Accuracy:", round(model_accuracy, 4))
print("Model Macro F1:", round(model_macro_f1, 4))

print("\nClassification Report:")
print(classification_report(y_test, model_pred))

Model Accuracy: 0.757
Model Macro F1: 0.6644

Classification Report:
              precision    recall  f1-score   support

        down       0.76      0.95      0.84      3252
        flat       0.59      0.49      0.53       231
         new       0.72      0.80      0.76       447
      stable       0.75      0.49      0.60      1192
          up       0.82      0.46      0.59       878

    accuracy                           0.76      6000
   macro avg       0.73      0.64      0.66      6000
weighted avg       0.76      0.76      0.74      6000



## 4. Errors and interpretation

### 4. Errors and Interpretation

I will examine the model's classification errors using the held-out test set. The analysis will focus on the confusion matrix, per-class performance, and examples of misclassified observations.

I will also inspect the learned Logistic Regression coefficients to identify which available features contribute most strongly to each outcome class.

The purpose of this analysis is not only to report the model score, but to understand where the model succeeds and fails and whether those errors reveal useful patterns for the Growth lane.

Any interpretation will be based only on information available at the observation point, without using future outcome fields as model inputs.


In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4.1 — Confusion matrix

from sklearn.metrics import confusion_matrix
import pandas as pd

labels = sorted(y_test.unique())

cm = confusion_matrix(
    y_test,
    model_pred,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{x}" for x in labels],
    columns=[f"Predicted_{x}" for x in labels]
)

print("Confusion Matrix:")
display(cm_df)

Confusion Matrix:


,Predicted_down,Predicted_flat,Predicted_new,Predicted_stable,Predicted_up
Actual_down,3080,49,49,44,30
Actual_flat,67,113,48,0,3
Actual_new,53,23,357,0,14
Actual_stable,526,7,29,589,41
Actual_up,314,0,13,148,403


In [29]:
# Section 4.2 — Inspect misclassified observations

test_results = X_test.copy()

test_results["actual"] = y_test.values
test_results["predicted"] = model_pred

misclassified = test_results[
    test_results["actual"] != test_results["predicted"]
].copy()

print("Total test rows:", len(test_results))
print("Misclassified rows:", len(misclassified))
print(
    "Error rate:",
    round(len(misclassified) / len(test_results), 4)
)

display(misclassified.head(20))

Total test rows: 6000
Misclassified rows: 1458
Error rate: 0.243


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,actual,predicted
25189,content_da57dd363452,client_19581e27de,320.0,0.82,HIGH,2.71,keyword article,informational,NaN,NaN,...,NaN,0.35,47.8,0.00,0.00,0.0,low,page_3_5,stable,up
6587,content_f55fda30b292,client_d029fa3a95,0.0,0.00,LOW,0.00,comparison article,informational,4285.0,29802.0,...,25000+,0.00,5.4,0.00,66.67,0.0,low,page_1,up,down
15532,content_e1db7ede3c55,client_8527a891e2,30.0,0.01,LOW,0.00,keyword article,informational,3368.0,20253.0,...,15000-25000,0.00,55.6,0.00,20.00,0.0,low,deep,stable,down
3341,content_e464c7eba756,client_d4735e3a26,NaN,NaN,NaN,NaN,feedly article,NaN,938.0,6669.0,...,<8000,0.00,6.0,0.00,0.00,0.0,low,page_1,flat,new
20646,content_e731548846d2,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,3258.0,21467.0,...,15000-25000,0.14,4.1,0.00,5.13,0.0,good,page_1,stable,down
4431,content_76c0dda8c65f,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,2717.0,16537.0,...,15000-25000,0.00,10.8,0.00,NaN,0.0,low,striking,stable,down
12514,content_cd49e62c96c3,client_2c624232cd,0.0,0.00,LOW,0.00,keyword article,informational,2179.0,14216.0,...,8000-15000,0.14,16.1,0.00,25.00,0.0,moderate,striking,down,stable
1111,content_94bfb13f7c09,client_f369cb89fc,20.0,0.94,HIGH,2.99,keyword article,commercial,2712.0,15927.0,...,15000-25000,1.05,6.0,0.00,33.33,0.0,low,page_1,stable,down
9337,content_11e822508b23,client_a88a7902cb,110.0,0.04,LOW,0.00,keyword article,transactional,3045.0,20068.0,...,15000-25000,0.00,3.4,20.00,36.36,0.0,low,page_1,flat,down
20218,content_4611d7b027c8,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,4617.0,30638.0,...,25000+,0.24,27.7,0.00,7.00,0.0,moderate,page_3_5,stable,down


In [30]:
# Section 4.3 — Interpret Logistic Regression coefficients

# Get the fitted preprocessing and classifier
fitted_preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

# Get transformed feature names
feature_names = fitted_preprocessor.get_feature_names_out()

# Create coefficient table
coef_df = pd.DataFrame(
    classifier.coef_,
    index=classifier.classes_,
    columns=feature_names
)

print("Model classes:")
print(classifier.classes_)

print("\nTop positive features for each class:")

for class_name in classifier.classes_:
    print(f"\n--- {class_name} ---")

    top_features = (
        coef_df.loc[class_name]
        .sort_values(ascending=False)
        .head(10)
    )

    display(
        top_features.to_frame("coefficient")
    )

Model classes:
['down' 'flat' 'new' 'stable' 'up']

Top positive features for each class:

--- down ---


,coefficient
numeric__impressions_prev_30d,23.320880
numeric__days_with_impressions,2.124288
numeric__clicks_prev_30d,1.629474
numeric__impressions_90d,1.442016
numeric__sessions_prev_30d,1.195787
categorical__client_id_client_3fdba35f04,1.111384
categorical__content_id_content_24796d98b025,0.954612
categorical__content_id_content_d19d9bb94617,0.890255
categorical__impression_tier_low,0.887230
categorical__content_id_content_a8cee66e4788,0.883698



--- flat ---


,coefficient
categorical__client_id_client_7f2253d7e2,3.119258
categorical__client_id_client_e629fa6598,1.175692
categorical__client_id_client_624b60c58c,1.079406
categorical__content_id_content_c8cc272bb4f6,0.979306
categorical__content_id_content_0a57fc95c688,0.966791
categorical__content_id_content_a6a337e6a4f7,0.949920
categorical__content_id_content_94b0ec4742b5,0.945298
categorical__content_id_content_4a9be0408cfe,0.939929
categorical__content_id_content_c857b94269c6,0.935694
categorical__content_id_content_e8c602e58a42,0.935627



--- new ---


,coefficient
numeric__impressions_last_30d,9.350413
categorical__client_id_client_25fc0e7096,2.055832
categorical__impression_tier_good,1.882726
numeric__sessions_last_30d,1.679120
categorical__client_id_client_f74efabef1,1.634645
numeric__clicks_last_30d,1.595680
categorical__client_id_client_434c9b5ae5,1.107819
categorical__content_id_content_9f9d01b5fedd,0.997198
categorical__content_id_content_c9fff7c4660c,0.995825
categorical__content_id_content_e848f7530bc6,0.980729



--- stable ---


,coefficient
numeric__days_with_impressions,2.700185
numeric__impressions_last_30d,2.628255
numeric__sessions_prev_30d,1.247132
categorical__impression_tier_low,0.990941
categorical__content_id_content_4f64fc6d61b4,0.989696
categorical__content_id_content_7906e807eb82,0.987037
categorical__content_id_content_0101e2578614,0.977951
categorical__content_id_content_141f8e20ddcb,0.977404
categorical__content_id_content_b4de3b03bb0b,0.971558
categorical__content_id_content_84e9d43ca93e,0.968124



--- up ---


,coefficient
numeric__impressions_last_30d,14.564903
numeric__days_with_impressions,2.165188
categorical__client_id_client_d59eced1de,1.038195
categorical__client_id_client_b4944c6ff0,0.970695
categorical__content_id_content_072ab0531040,0.970678
categorical__content_id_content_41405e2a2f88,0.933159
categorical__content_id_content_439cddfd5484,0.929012
categorical__content_id_content_51715e336a42,0.912620
categorical__content_id_content_262ae9dd701a,0.908924
categorical__content_id_content_32b4a4b67d63,0.903518


In [31]:
# Section 4.4 — Top negative features for each class

print("Top negative features for each class:")

for class_name in classifier.classes_:
    print(f"\n--- {class_name} ---")

    bottom_features = (
        coef_df.loc[class_name]
        .sort_values(ascending=True)
        .head(10)
    )

    display(
        bottom_features.to_frame("coefficient")
    )

Top negative features for each class:

--- down ---


,coefficient
numeric__impressions_last_30d,-26.425072
numeric__clicks_last_30d,-1.507738
categorical__client_id_client_25fc0e7096,-0.844878
categorical__content_id_content_e640070cf164,-0.815682
categorical__content_id_content_324542fecc93,-0.775819
categorical__content_id_content_54d7514e6b2c,-0.766375
categorical__content_id_content_77914166676c,-0.757277
categorical__content_id_content_75568063406d,-0.753479
categorical__content_id_content_4f166f39b2b5,-0.753042
categorical__content_id_content_2eceefd208e2,-0.752179



--- flat ---


,coefficient
numeric__days_with_impressions,-4.180722
categorical__freshness_tier_0-30,-1.295705
numeric__sessions_last_30d,-1.136665
categorical__client_id_client_d4735e3a26,-1.029728
categorical__client_id_client_3fdba35f04,-1.026513
categorical__client_id_client_434c9b5ae5,-0.951503
categorical__client_id_client_6208ef0f77,-0.927329
categorical__content_type_comparison article,-0.897519
categorical__provider_used_google,-0.847732
categorical__client_id_client_25fc0e7096,-0.810858



--- new ---


,coefficient
numeric__impressions_prev_30d,-7.528789
numeric__days_with_impressions,-2.808939
numeric__sessions_prev_30d,-2.288637
categorical__impression_tier_low,-1.890891
numeric__clicks_prev_30d,-1.887684
categorical__client_id_client_7f2253d7e2,-1.173958
numeric__impressions_90d,-1.144020
categorical__client_id_client_9400f1b21c,-1.096597
categorical__content_id_content_24796d98b025,-0.902879
categorical__content_id_content_4f64fc6d61b4,-0.894415



--- stable ---


,coefficient
categorical__client_id_client_7f2253d7e2,-1.111993
categorical__client_id_client_d59eced1de,-0.976881
categorical__content_id_content_23d452af4198,-0.805356
categorical__content_id_content_f42eb861c6dd,-0.673064
categorical__client_id_client_8722616204,-0.661438
numeric__impressions_prev_30d,-0.658875
categorical__content_id_content_c1ee712bc230,-0.649597
categorical__content_id_content_d0513fb2a904,-0.648904
categorical__content_id_content_d94dc4707433,-0.614854
categorical__content_id_content_6101db1e202c,-0.604009



--- up ---


,coefficient
numeric__impressions_prev_30d,-15.052938
categorical__client_id_client_7f2253d7e2,-1.262623
categorical__content_id_content_9f9d01b5fedd,-0.996922
categorical__content_id_content_aef984d6c132,-0.962926
categorical__content_id_content_6a4f086ef3c4,-0.893216
numeric__impressions_90d,-0.875374
categorical__content_id_content_d115b5fa1a44,-0.692640
categorical__client_id_client_f74efabef1,-0.689513
categorical__client_id_client_e629fa6598,-0.669248
numeric__users_90d,-0.631563


In [32]:
# Section 4.5 — Error summary by actual class

error_summary = (
    test_results.assign(
        error=test_results["actual"] != test_results["predicted"]
    )
    .groupby("actual")["error"]
    .agg(
        total_rows="count",
        errors="sum",
        error_rate="mean"
    )
)

error_summary["error_rate"] = error_summary["error_rate"].round(4)

print("Error summary by actual class:")
display(error_summary)

Error summary by actual class:


,total_rows,errors,error_rate
actual,,,
down,3252,172,0.0529
flat,231,118,0.5108
new,447,90,0.2013
stable,1192,603,0.5059
up,878,475,0.5410


### Final interpretation

The held-out test results show where the Logistic Regression model performs well and where it makes classification errors across the observed outcome classes.

The confusion matrix and error summary are used to identify classes that are easier or harder for the model to distinguish. Misclassified observations are treated as evidence of uncertainty or overlapping patterns in the available observation-point features rather than as proof that a particular feature causes the outcome.

The coefficient analysis provides directional information about which available features the model uses when distinguishing between the outcome classes. These coefficients describe model associations and should not be interpreted as causal effects.

Overall, the error analysis is used to determine whether the learned model provides useful predictive signal and to identify areas where additional features or a different modeling approach may be worth investigating.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.